# Module 6: Agentic Search

⏱ **Estimated time:** 25 minutes

In this notebook, you'll build an agent with four search tools using the [Strands Agents SDK](https://strandsagents.com/), then score it on the same 50 questions as every other configuration.

## What You'll Do

1. Write three search functions that do things RAG can't
2. Turn them into agent tools, alongside RAG with reranking
3. Watch the agent investigate the s1043A penalty question
4. Score the agent on the 50 questions
5. Compare it with every other configuration, question by question

---
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.  
SPDX-License-Identifier: MIT-0

## Step 1: Install the Dependencies

Run this first to install the required packages for the notebook.

In [ ]:
# Install the notebook's dependencies (requirements-notebooks.txt sits beside the notebook)
get_ipython().run_line_magic("pip", "install -q -r requirements-notebooks.txt")
print("✅ Dependencies installed.")

## Step 2: Configuration

Sets up the variables and AWS resources used in this notebook.

In [ ]:
# Settings
import asyncio
import boto3
import json
import logging
import os
import re
from datetime import datetime, timezone

# Let Inspect AI run inside Jupyter
import nest_asyncio
nest_asyncio.apply()
# Hide noisy asyncio warnings
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

# The AWS region, and which dataset to load
REGION = os.environ.get("AWS_REGION", "us-east-1")
DOMAIN = "legal"
JURISDICTION = "au"

# Which version of the 50-question evaluation set to use, and how many questions to score
EVAL_VERSION = "v1"       # "v1" matches every other module
NUM_EVAL_RECORDS = 50
MAX_CONNECTIONS = 8       # parallel requests to Bedrock

# The untuned model that answers the questions
BASELINE_MODEL = "us.amazon.nova-micro-v1:0"
# Where the evaluation questions live in S3 (v1 has no version suffix in its filename)
EVAL_KEY = (f"{DOMAIN}/eval-{DOMAIN}-{JURISDICTION}.jsonl" if EVAL_VERSION == "v1"
            else f"{DOMAIN}/eval-{DOMAIN}-{JURISDICTION}-{EVAL_VERSION}.jsonl")
CORPUS_PREFIX = f"{DOMAIN}/corpus/"   # the same prefix the knowledge base reads
# Where each configuration's score is saved, so later cells and modules can compare them
CELL_RESULTS_PREFIX = "results/cells/"
NOTEBOOK_NAME = "Module-6-Agentic-Search"
# Semantic search fetches 50 candidates, then a reranking model keeps the best few
RETRIEVE_WIDE = 50
RERANK_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/cohere.rerank-v3-5:0"

print(f"EVAL_VERSION={EVAL_VERSION}")

Reads the bucket names and knowledge base ID from the stack outputs.

> If something seems missing later on, check the account and region printed here first.

In [ ]:
# AWS clients, and bucket names and the knowledge base ID from the stack outputs
def stack_outputs():
    """Every CloudFormation output in the region, by key."""
    # The CloudFormation stack records the names of the resources it created
    cf = boto3.client("cloudformation", region_name=REGION)
    return {o["OutputKey"]: o["OutputValue"]
            for page in cf.get_paginator("describe_stacks").paginate()
            for s in page["Stacks"] for o in s.get("Outputs", [])}

# Look up the S3 buckets and the Bedrock knowledge base (optional, so it may be missing)
outputs = stack_outputs()
DATASET_BUCKET = outputs["DatasetBucketName"]
RESULTS_BUCKET = outputs["ResultsBucketName"]
KNOWLEDGE_BASE_ID = outputs.get("KnowledgeBaseId")

# Clients for S3 and Bedrock
s3 = boto3.client("s3", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)

print(f"KnowledgeBaseId: {KNOWLEDGE_BASE_ID}")
# Without a knowledge base there is nothing for meaning-based search to query
if not KNOWLEDGE_BASE_ID:
    print("⚠️  No knowledge base found, so semantic_search is turned off and the agent has three tools.")

### Step 2a: The Model Under Test

The cell below sets the model being assessed, untuned Nova Micro. Keeping the
model fixed means any change in score comes from the search, not the model.

In [ ]:
# The model under test: untuned Nova Micro
# The label tags saved results as "base" (untuned) rather than "finetuned"
MODEL_LABEL = "base"
# Inspect AI names Bedrock models with a "bedrock/" prefix
MODEL_UNDER_TEST = f"bedrock/{BASELINE_MODEL}"
print(f"Model under test: {MODEL_UNDER_TEST}")

### Step 2b: Configure the Rubric and Judge

The cell below sets up the rubric and the judge model that grades each answer. This is the same configuration as Module 5, to ensure direct comparability.

In [ ]:
# The rubric and judge
# Inspect AI is an open-source framework for running and scoring model evaluations
# A solver produces the answer to each question, and a scorer grades it
from inspect_ai import Task, task, eval as inspect_eval
from inspect_ai.dataset import FieldSpec, json_dataset
from inspect_ai.model import get_model, GenerateConfig
from inspect_ai.solver import generate, solver, Generate, TaskState
from inspect_ai.scorer import model_graded_qa

# The judge's rubric, identical in Modules 2, 4, 5 and 6
LEGAL_INSTRUCTIONS = """Grade the submitted answer against the reference answer for a question about Australian law.

Mark the answer CORRECT only if ALL of the following hold:
- It reaches the correct legal conclusion (e.g. liable/not liable, valid/void).
- It cites the correct legislation or section where the reference answer does.
- It is specific to Australian law, not generic or another jurisdiction's law.

Mark it INCORRECT if the legal conclusion is wrong, the cited provision is wrong
or fabricated, or the answer is generic rather than Australian-specific.

First explain your reasoning, then end with your grade in exactly this format:
"GRADE: C" if correct or "GRADE: I" if incorrect."""

# The judge model
# A judge model is a stronger model that grades each answer against the reference answer, using the rubric
# Temperature 0 makes the grading as repeatable as possible
grader = get_model("bedrock/us.anthropic.claude-sonnet-4-6",
                   config=GenerateConfig(temperature=0.0))

print("✓ Rubric and judge pinned (Sonnet 4.6, temperature 0)")

**Define the prompt.** `build_grounded_prompt` builds the prompt the model answers
from, using the passages found. `as_span` puts the tools' results into the same
format as knowledge base results, so both can go into the same prompt.

In [ ]:
# Build the answering prompt, the same as Module 5's
def build_grounded_prompt(chunks, query):
    """Wrap retrieved spans and the question into the graded prompt."""
    # Join the text of every retrieved passage, with a divider between them
    context = "\n\n---\n\n".join(c["content"]["text"] for c in chunks)
    # Put the legislation inside tags so the model can tell it apart from the question
    return (
        "Use the following Australian legislation extracts to answer the "
        "question. If they are relevant, cite the specific section.\n\n"
        f"<legislation>\n{context}\n</legislation>\n\n"
        f"Question: {query}"
    )


def as_span(text, source):
    """Wrap plain text in the chunk shape build_grounded_prompt expects."""
    # Same shape as a knowledge base result, so agent evidence and RAG results are handled alike
    return {"content": {"text": text}, "location": {"s3Location": {"uri": source}}}

print("✅ Ready to proceed!")

**Define how results are saved and loaded.** The cell below saves each score to
`results/cells/` in S3 and loads earlier ones. Only full 50-question runs are
loaded; shorter test runs are ignored.

In [ ]:
# Save and load scores in results/cells/
def save_cell_result(cell, model_label, model_id, score, per_item, config=None):
    """Save one configuration's score to results/cells/."""
    # One record per run: the overall score, the per-question results, and how it was produced
    rec = {"cell": cell, "model_label": model_label, "model_id": model_id, "score": score,
           "n": len(per_item), "judge": "us.anthropic.claude-sonnet-4-6",
           "eval_key": EVAL_KEY, "notebook": NOTEBOOK_NAME,
           "timestamp": datetime.now(timezone.utc).isoformat(),
           "config": config or {}, "per_item": per_item}
    # A timestamp in the filename means each run is kept rather than overwritten
    key = f"{CELL_RESULTS_PREFIX}{cell}-{model_label}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}.json"
    s3.put_object(Bucket=RESULTS_BUCKET, Key=key, Body=json.dumps(rec).encode(),
                  ContentType="application/json")
    print(f"  ✓ saved s3://{RESULTS_BUCKET}/{key}")

def load_cell_results():
    """Newest full-size (50-item) result for each configuration."""
    newest = {}
    # Read every saved result and keep the newest one for each (configuration, model) pair
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=RESULTS_BUCKET, Prefix=CELL_RESULTS_PREFIX)
    for obj in (o for page in pages for o in page.get("Contents", []) if o["Key"].endswith(".json")):
        rec = json.loads(s3.get_object(Bucket=RESULTS_BUCKET, Key=obj["Key"])["Body"].read())
        key = (rec.get("cell"), rec.get("model_label"))
        # Skip partial runs with fewer than 50 questions
        if rec.get("n", 0) >= NUM_EVAL_RECORDS and rec["timestamp"] > newest.get(key, {}).get("timestamp", ""):
            newest[key] = rec
    return newest

print("✅ Ready to proceed!")

**Define the scoring function.** `score_cell` asks the model each of the 50
questions, has the judge grade every answer, and returns the overall score and the
result for each question.

In [ ]:
# Score one configuration on the 50 questions and save the result
def score_cell(cell, task_solver=None, *, config=None, save=True, model=None):
    """Score one configuration on the 50 questions. Returns (score, per_item)."""
    under_test = model or MODEL_UNDER_TEST
    # A task bundles the dataset, the solver that produces each answer, and the scorer that grades it
    # @task registers the function with Inspect AI
    @task
    def cell_task():
        return Task(
            # Each question's prompt is the input, and the reference answer is what the judge compares against
            dataset=json_dataset(f"s3://{DATASET_BUCKET}/{EVAL_KEY}",
                                 FieldSpec(input="prompt", target="referenceResponse", metadata=["category"]),
                                 limit=NUM_EVAL_RECORDS),
            # Without a custom solver, the model just answers the question on its own
            solver=task_solver or generate(),
            # model_graded_qa asks the judge model to grade each answer using the rubric
            scorer=model_graded_qa(instructions=LEGAL_INSTRUCTIONS, model=grader),
        )

    # Run all 50 questions, up to MAX_CONNECTIONS at a time, and keep the run's log
    log = inspect_eval(cell_task(), model=under_test, log_dir=f"./inspect-logs/{cell}-{MODEL_LABEL}",
                       max_tokens=1024, max_connections=MAX_CONNECTIONS, temperature=0.0)[0]
    if log.status != "success" or not log.samples:
        print(f"  ✗ {cell}: the evaluation failed. Check the log above.")
        return None, []

    # Turn each question's grade into 1 (correct, "C") or 0 (incorrect)
    per_item = [{"id": s.id if s.id is not None else i,
                  "category": (s.metadata or {}).get("category", "unknown"),
                  "score": 1.0 if s.scores and str(next(iter(s.scores.values())).value).upper().startswith("C") else 0.0}
                for i, s in enumerate(log.samples, 1)]
    # The overall score is the fraction answered correctly
    score = sum(p["score"] for p in per_item) / len(per_item)
    print(f"  {cell} ({MODEL_LABEL}): {score:.3f}  (n={len(per_item)})")
    if save:
        save_cell_result(cell, MODEL_LABEL, under_test, score, per_item, config)
    return score, per_item

def print_comparison(cell_order):
    """Print the newest score for each configuration, base and fine-tuned."""
    latest = load_cell_results()
    print(f"  {'Configuration':<36} {'Base':>8} {'Fine-tuned':>12}")
    print("  " + "-" * 58)
    # One row per configuration, with a dash where a model has not been scored yet
    for cell, label in cell_order:
        b, f = (latest.get((cell, m), {}).get("score") for m in ("base", "finetuned"))
        print(f"  {label:<36} {f'{b:.3f}' if b is not None else '—':>8} {f'{f:.3f}' if f is not None else '—':>12}")
    print("\n  — = not measured yet. Repeat runs vary by about 0.04.")
    return latest

print("✅ Ready to proceed!")

**Define the paired comparison.** At 50 questions, two total scores a few points
apart may just be run-to-run variation. Comparing question by question tells you
more. `paired_compare` counts how many questions a configuration **fixed** (wrong
before, right after) and **broke** (right before, wrong after), and `mcnemar_exact`
tests whether that split could be chance.

In [ ]:
# Compare two runs question by question, with an exact McNemar test
# The McNemar test checks whether two runs on the same questions really differ,
# using only the questions where they disagree (one right, the other wrong)
def mcnemar_exact(fixed, broken):
    """Two-sided exact McNemar p-value for a paired comparison."""
    from math import comb

    n = fixed + broken
    # If the runs never disagree, there is no evidence of a difference
    if n == 0:
        return 1.0
    # By chance, each disagreement is a coin flip. Add up the odds of a split
    # at least this lopsided, in either direction
    k = min(fixed, broken)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / (2 ** n))


def paired_compare(baseline_per_item, variant_per_item):
    """Item-level diff between two scored runs. Returns counts plus the p-value."""
    # Match the two runs by question ID, comparing only questions both runs answered
    base = {str(p["id"]): p for p in baseline_per_item}
    var = {str(p["id"]): p for p in variant_per_item}
    shared = sorted(set(base) & set(var))

    # Sort each question into one of four groups by whether each run got it right
    fixed, broken, both_right, both_wrong = [], [], [], []
    for i in shared:
        b, v = base[i]["score"], var[i]["score"]
        if b == 0 and v == 1:
            fixed.append(i)
        elif b == 1 and v == 0:
            broken.append(i)
        elif b == 1:
            both_right.append(i)
        else:
            both_wrong.append(i)

    # delta is the change in score across the shared questions
    return {
        "n_compared": len(shared),
        "n_baseline_only": len(set(base) - set(var)),
        "n_variant_only": len(set(var) - set(base)),
        "fixed": fixed,
        "broken": broken,
        "both_right": both_right,
        "both_wrong": both_wrong,
        "delta": (len(fixed) - len(broken)) / len(shared) if shared else 0.0,
        "p_value": mcnemar_exact(len(fixed), len(broken)),
    }


print("✓ Harness ready (judge: Sonnet 4.6, same rubric as Modules 2, 4 and 5)")

**Load earlier results.** The cell below lists the scores already saved in S3 for
the other configurations: no retrieval, RAG, and RAG with reranking. Step 10 puts
them in a table next to this notebook's result.

> A configuration that hasn't been run yet shows as `—` in the Step 10 table.

In [ ]:
# Print the results already saved by earlier modules
# Each configuration's newest full-size score, from results/cells/ in S3
prior = load_cell_results()
for (cell, label), rec in sorted(prior.items()):
    print(f"  {label:<10} {cell:<16} {rec['score']:.3f}  ({rec['timestamp'][:16]})")
if not prior:
    print("No earlier results yet. Run Modules 2 and 5 first.")

## Step 3: Read the Corpus from S3

The tools read the legislation from the same S3 prefix the knowledge base uses,
so every tool sees the same text. It's about 15 MB, loaded once.

**First, a trap in the source text.** Two quirks would make exact-phrase search
silently find nothing:

1. **Non-breaking spaces.** legislation.gov.au uses them so section references
   don't wrap across lines, so a search for `"Subsection 1043A(1)"` typed with an
   ordinary space matches nothing.
2. **Missing spaces after subsection numbers**, such as `"(1)If"` instead of
   `"(1) If"`.

`normalise` fixes both.

In [ ]:
# Normalise the text so exact-phrase search works
import re
import unicodedata

# Whitespace to collapse, written as escapes
# Matches runs of spaces, tabs and Unicode space characters such as non-breaking spaces
_WS = re.compile("[ \\t\\u00a0\\u202f\\u2000-\\u200a\\u2028\\u2029\\u3000\\u180e\\ufeff]+")

# Adds the missing space in "(1)If" and "11.1If"
# Matches a marker such as (1), (a) or 11.1 that runs straight into a letter
_SUBSEC_RUNON = re.compile(r"(\([0-9a-z]+\)|\d+\.\d+)(?=[A-Za-z])")


def normalise(text):
    """Collapse whitespace and unglue subsection markers, preserving newlines."""
    # NFKC converts look-alike Unicode characters (such as full-width digits) to their plain forms
    text = unicodedata.normalize("NFKC", text)
    out = []
    # Clean each line separately so line breaks, and so line numbers, stay the same
    for line in text.split("\n"):
        out.append(_SUBSEC_RUNON.sub(r"\1 ", _WS.sub(" ", line)).strip())
    return "\n".join(out)


print("✓ Normalisation ready")

**Fetching and indexing.** `fetch_corpus_from_s3` downloads the Acts, `build_index`
makes them searchable, and `pick_acts` turns a hint like `"corporations"` into the
matching Act. An unrecognised hint searches every Act rather than none.

In [ ]:
# Load the Acts from S3 and index them
def build_index(documents):
    """Turn {filename: raw text} into the searchable corpus the tools use."""
    corpus = {}
    for filename, raw in sorted(documents.items()):
        text = normalise(raw)
        # The slug is the filename without its folder or .txt ending, used as the Act's short ID
        slug = filename.rsplit("/", 1)[-1].removesuffix(".txt")
        # Use the title from a SOURCE: header on the first line if there is one, otherwise build it from the slug
        first = text.split("\n", 1)[0]
        title = slug.replace("-", " ").title()
        if first.startswith("SOURCE:"):
            title = first[len("SOURCE:"):].split("|")[0].strip() or title
        corpus[slug] = {"title": title, "text": text, "chars": len(text)}
    return corpus


def fetch_corpus_from_s3(bucket, prefix, client=None):
    """Download every .txt under the prefix. Returns {key: text}."""
    cli = client or s3
    docs, token = {}, None
    # S3 lists at most 1,000 objects per call, so keep going while there are more pages
    while True:
        kw = {"Bucket": bucket, "Prefix": prefix}
        if token:
            kw["ContinuationToken"] = token
        page = cli.list_objects_v2(**kw)
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".txt"):
                body = cli.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read()
                # Replace any bytes that are not valid UTF-8 rather than failing
                docs[obj["Key"]] = body.decode("utf-8", errors="replace")
        if not page.get("IsTruncated"):
            break
        token = page.get("NextContinuationToken")
    return docs


def pick_acts(corpus, act=None):
    """Resolve an Act hint to slugs. Substring match on slug or title, else all."""
    if not act:
        return list(corpus)
    # Match ignoring case and hyphens, so "criminal code" finds "criminal-code"
    needle = normalise(act).lower().replace("-", " ")
    hits = [s for s, d in corpus.items()
            if needle in s.replace("-", " ").lower() or needle in d["title"].lower()]
    # If nothing matches, search every Act rather than return nothing
    return hits or list(corpus)

print("✅ Ready to proceed!")

**Load the corpus.**

> This should show 12 Acts and 0 non-breaking spaces remaining.

In [ ]:
# Fetch and index the corpus, then report what arrived
CORPUS = build_index(fetch_corpus_from_s3(DATASET_BUCKET, CORPUS_PREFIX))
# Count leftover non-breaking spaces, which would stop exact-phrase search from matching
nbsp = sum(d["text"].count("\u00a0") for d in CORPUS.values())

print(f"✓ {len(CORPUS)} Acts, {sum(d['chars'] for d in CORPUS.values()) / 1e6:.1f}M characters")
# List each Act with its size in thousands of characters
for d in sorted(CORPUS.values(), key=lambda d: d["title"]):
    print(f"  {d['chars'] / 1000:>6,.0f}k  {d['title']}")
print(f"\nNon-breaking spaces left after normalise(): {nbsp} (expected 0)")
if len(CORPUS) < 12:
    print(f"⚠️  Only {len(CORPUS)} of 12 Acts are in S3. Load the data first (Prerequisite Setup).")

## Step 4: Write the Search Functions

The cells below define three search functions in plain Python: look up a section
by number, search for an exact phrase, and read the lines around a result.

**`find_section`: look up a section by number.** The cell below finds a section by
matching its heading, such as `180`. It matches the whole number, so `18` doesn't
match `180`, and allows a full stop after the number, as in the Constitution's
`92. Trade within...`.

In [ ]:
# find_section: look a section up by number
def find_section(corpus, section, act=None, max_chars=4000):
    """STRUCTURAL: look a provision up by its exact section number."""
    # Matches a line that starts with the section number (e.g. "180"), an optional full stop, then the heading
    pattern = re.compile(
        rf"^{re.escape(normalise(section).strip().upper())}[.\u3002]?\s+(?P<heading>\S.*)$",
        re.MULTILINE)

    results = []
    # Look in the named Act, or in every Act if none is given
    for slug in pick_acts(corpus, act):
        text = corpus[slug]["text"]
        for m in pattern.finditer(text):
            start = m.start()
            # The section ends where the next section number starts
            # Matches the next line that starts with a section number, e.g. "181 " or "1043B. "
            nxt = re.search(r"^\d+[A-Z]*[.\u3002]?\s+\S", text[m.end():], re.MULTILINE)
            # If there is no next section, stop at max_chars
            end = m.end() + nxt.start() if nxt else start + max_chars
            results.append({
                "act": slug,
                "title": corpus[slug]["title"],
                "section": normalise(section).strip(),
                "heading": m.group("heading").strip(),
                "text": text[start:min(end, start + max_chars)].strip(),
                # start_line counts the line breaks before the match, so read_around can jump to it
                "start_line": text.count("\n", 0, start) + 1,
            })
    return results

print("✅ Ready to proceed!")

**`search_text`: search for an exact phrase.** The cell below finds every line that
contains a phrase, such as `Subsection 1043A(1)`. For each match, it also returns
the lines after it, because in a penalty table the penalty is on the next line. It
returns matches from across the whole Act, not just the first few, because the
penalty table is at the end.

In [ ]:
# search_text: find an exact phrase, with the lines that follow it
def search_text(corpus, query, act=None, max_hits=8, window=700):
    """KEYWORD: find a literal string, with the following lines included."""
    # Normalise the query the same way as the corpus, then match ignoring case
    needle = normalise(query).strip()
    if not needle:
        return []
    lowered = needle.lower()

    # Find every match, one per line
    found = []
    for slug in pick_acts(corpus, act):
        text = corpus[slug]["text"]
        hay = text.lower()
        pos = hay.find(lowered)
        # Track the line number of each match, and record only the first match on each line
        seen_lines, line_no, scanned = set(), 1, 0
        while pos != -1:
            line_no += text.count("\n", scanned, pos)
            scanned = pos
            if line_no not in seen_lines:
                seen_lines.add(line_no)
                found.append((slug, pos))
            pos = hay.find(lowered, pos + len(needle))
            if len(found) > 5000:        # guard against huge result sets
                break

    # Keep matches spread across the Act, not just the first few
    total = len(found)
    if total > max_hits:
        # Pick max_hits matches evenly spaced through the full list
        step = (total - 1) / (max_hits - 1) if max_hits > 1 else 0
        found = [found[i] for i in sorted({round(i * step) for i in range(max_hits)})]

    # Each result includes a third of the window before the match and the full window after it,
    # so a penalty on the line after a table row is included
    results = []
    for n, (slug, pos) in enumerate(found, 1):
        text = corpus[slug]["text"]
        results.append({
            "act": slug,
            "title": corpus[slug]["title"],
            "match": text[pos:pos + len(needle)],
            "text": text[max(0, pos - window // 3):
                         min(len(text), pos + len(needle) + window)].strip(),
            "start_line": text.count("\n", 0, pos) + 1,
            "hit": n,
            "of": total,          # total matches
        })
    return results

print("✅ Ready to proceed!")

**`read_around`: read the lines around a result.** The cell below returns the text
around a given line number. The agent uses it to read more of a section, or to
follow a reference such as *"see section 1317E"*.

In [ ]:
# read_around: read the lines around a position
def read_around(corpus, act, line, before=10, after=40):
    """NAVIGATION: read a line range, to follow a cross-reference."""
    # Use the first Act that matches the name
    slugs = pick_acts(corpus, act)
    if not slugs:
        return None
    slug = slugs[0]
    lines = corpus[slug]["text"].split("\n")
    # Line numbers start at 1, so convert to list positions and clip to the start and end of the Act
    lo = max(0, int(line) - 1 - before)
    hi = min(len(lines), int(line) - 1 + after)
    return {
        "act": slug,
        "title": corpus[slug]["title"],
        "from_line": lo + 1,
        "to_line": hi,
        "text": "\n".join(lines[lo:hi]).strip(),
    }


print("✓ Three retrieval functions defined — plain Python, no AWS calls.")
print("  Step 6 turns them into agent tools by adding @tool.")

## Step 5: Test the Search Functions

The cell below runs each search function on an example. It makes no model calls,
so it costs nothing.

> Check that `Subsection 1043A(1)` comes back with `15 years imprisonment` on the
> next line, and note that it isn't the first match.

In [ ]:
# Test the three functions on s180 and the s1043A penalty row
# Look up section 180 (directors' duty of care and diligence) in the Corporations Act
s180 = find_section(CORPUS, "180", act="corporations")[0]
print(f"find_section('180'): s{s180['section']} {s180['heading']} (line {s180['start_line']})")
# chr(10) is a newline; replace it so the excerpt prints on one line
print(f"   {s180['text'][:200].replace(chr(10), ' / ')}...")

# Find every mention of "Subsection 1043A(1)", and pick out the one in the penalty table
hits = search_text(CORPUS, "Subsection 1043A(1)", act="corporations")
penalty = next((h for h in hits if "15 years imprisonment" in h["text"]), None)
print(f"\nsearch_text('Subsection 1043A(1)'): {hits[0]['of']} matches, {len(hits)} returned")
for h in hits:
    print(f"   match {h['hit']:>2} of {h['of']}, line {h['start_line']:>6}"
          f"{'  ← the penalty row' if h is penalty else ''}")
if penalty:
    i = penalty["text"].find("Subsection 1043A(1)")
    print(f"   The row: {penalty['text'][i:i + 62]!r}")

# Find the first mention of section 1317E, then read the lines around it
ref = search_text(CORPUS, "1317E", act="corporations", max_hits=1)[0]
nearby = read_around(CORPUS, "corporations", ref["start_line"], before=1, after=4)
print(f"\nread_around(line {ref['start_line']}): {nearby['text'][:160].replace(chr(10), ' / ')}...")

## Step 6: Turn Them into Agent Tools

With Strands, turning a Python function into a tool the agent can use takes one
line: add the `@tool` decorator. Strands reads the function's type hints and
docstring and describes the tool to the model for you, so there's no schema to
write by hand. For example:

```python
@tool
def find_section(section: str, act: str = "") -> str:
    """Look up a provision by its exact section number."""
    ...
```

The model only sees the docstring and arguments, so the docstring is what tells
the agent when to use the tool. To give the agent a new ability, write a function,
add `@tool`, and pass it to the agent.

First, the settings:

- **`ORCHESTRATOR_MODEL`** runs the agent. It's Nova Pro, not the model under test,
  because Nova Micro is poor at using tools. This is recorded with the result.
- **The final answer** is written by `MODEL_UNDER_TEST` (Nova Micro), so any
  difference in score comes from the search.
- **`AGENT_CONTEXT_CHARS`** caps the evidence in the answering prompt at about
  the same size as RAG's, so answering costs about the same.

In [ ]:
# Agent settings
# Strands Agents is an open-source framework for building AI agents. An agent is a model
# that can call tools (Python functions) in a loop until it has what it needs
from strands import Agent, tool
from strands.models import BedrockModel
from strands.types.agent import Limits

ORCHESTRATOR_MODEL = "us.amazon.nova-pro-v1:0"   # runs the agent

# Limits on how much work the agent does per question, and how much text it collects
AGENT_MAX_TURNS = 10           # most model calls per question
AGENT_SEARCH_TOP_K = 5         # passages per semantic_search
AGENT_KEYWORD_HITS = 6         # matches per search_text
AGENT_EXCERPT_CHARS = 700      # how much of each result the agent sees
AGENT_CONTEXT_CHARS = 36_000   # answering-prompt budget, matched to Module 5

print(f"Orchestrator: {ORCHESTRATOR_MODEL}")
print(f"Answerer:     {MODEL_UNDER_TEST}")

**The system prompt tells the agent which search suits which question.** It
doesn't tell it what to call. The key instructions:

- **If a section number is named, call `find_section` first**, because searching by
  meaning returns look-alike sections.
- **If the question asks about a penalty, also call `search_text`**, because
  penalties sit in a table at the end of the Act.
- **Spell abbreviations out as the legislation does.** "APP 11" finds nothing;
  "Australian Privacy Principle 11" does.

In [ ]:
# The agent's instructions
# The system prompt tells the agent which tool suits which kind of question, and to gather evidence only
AGENT_SYSTEM_PROMPT = """You are a legal research assistant working on a corpus of Australian legislation (Commonwealth and NSW Acts).

Your ONLY job is to FIND the provisions needed to answer the question. You must NOT answer the legal question yourself.

You have four tools, and choosing the right one matters more than wording a query well:

- find_section(section, act) - EXACT lookup by section number. Use this FIRST whenever the question names a section. Australian Acts impose near-identical duties on many different officeholders, so a semantic search for "section 180" returns ten provisions that merely sound like it; find_section returns the real one.
- search_text(query, act) - LITERAL string search. Use it for anything tabular or structural rather than prose: penalties, maximum terms, dollar amounts, defined terms, and cross-references. It matches EXACT text, so spell abbreviations out the way the legislation does: search "Australian Privacy Principle 11", not "APP 11"; "Australian Consumer Law", not "ACL". If a search returns nothing, that usually means your wording differs from the Act's, not that the provision is absent - reformulate before giving up. Penalties usually live in a Schedule penalty table at the END of an Act, written as a bare row - "Subsection 1043A(1)" on one line and the penalty on the next - far from the section creating the offence. Semantic search cannot find those; this can.
- semantic_search(query) - meaning-based search. Best when the question describes a concept without naming a provision.
- read_around(act, line) - read the surrounding lines of a hit. Use it to follow a cross-reference, e.g. a provision that says "this is a civil penalty provision (see section 1317E)".

Method:
1. If a section number is named, call find_section first.
2. If the question asks about a penalty, maximum term, or amount, ALSO call search_text - the answer is almost never in the section itself.
3. If you only have a concept, start with semantic_search, then pin down the exact provision with find_section once you know its number.
4. Follow a cross-reference with read_around when the provision you found points elsewhere.

Stop as soon as you have the provisions. Then reply with ONE short sentence naming what you found. Do not quote the legislation back, and do not answer the question."""

print("✅ Ready to proceed!")

**Define reranking for `semantic_search`.** The cell below retrieves candidate
passages from the knowledge base and reranks them with Cohere Rerank 3.5, the same
retrieve-then-rerank approach as RAG with reranking.

In [ ]:
# Rerank passages, as in Module 5
# A reranker reads the query and each passage together and scores how relevant the passage is.
# This is slower than vector search but more accurate, so it reorders vector search results
def rerank_candidates(rt, query, results, top_k):
    """Reorder vector candidates with a cross-encoder, keep the best top_k."""
    # Send the query and each candidate passage (cut to 4,000 characters) to the Cohere rerank model on Bedrock
    out = rt.rerank(
        queries=[{"type": "TEXT", "textQuery": {"text": query}}],
        sources=[{"type": "INLINE",
                  "inlineDocumentSource": {"type": "TEXT",
                                           "textDocument": {"text": r["content"]["text"][:4000]}}}
                 for r in results],
        rerankingConfiguration={
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "numberOfResults": min(top_k, len(results)),
                "modelConfiguration": {"modelArn": RERANK_MODEL_ARN},
            },
        },
    )
    # Keep the passages in their new order, and note where each one ranked before reranking
    kept, moves = [], []
    for new_rank, item in enumerate(out["results"], 1):
        kept.append(results[item["index"]])
        moves.append({"new_rank": new_rank, "vector_rank": item["index"] + 1,
                      "relevance": round(item.get("relevanceScore", 0.0), 4)})
    return kept, moves

print("✅ Ready to proceed!")

**Define the tools.** Each Step 4 function gets a thin `@tool` wrapper that formats
results for the agent, records the call, and keeps the full text for the
answering prompt while the agent sees a short excerpt.

- **`name=`** makes each tool's name match the one the system prompt uses.
- **Each question gets its own evidence store**, so evidence from one question
  can't leak into another's answer when they run in parallel.

In [ ]:
# Wrap the search functions as agent tools
def build_toolset(kb_id, region, corpus, evidence, calls):
    """Wrap the Step 4 functions as agent tools, writing into an evidence store."""
    # Client for the Bedrock knowledge base and reranking APIs
    rt = boto3.client("bedrock-agent-runtime", region_name=region)

    def keep(text, source):
        """Full text aside for the answerer; the agent only sees an excerpt."""
        # Store each passage once, keyed by its text, so repeats are dropped
        if text and text not in evidence:
            evidence[text] = as_span(text, source)

    # The @tool decorator turns a Python function into a tool the agent can call. Strands reads
    # the type hints and docstring to tell the agent what the tool does and what arguments it takes
    # name= matches the tool names in the system prompt
    @tool(name="find_section")
    def find_section_tool(section: str, act: str = "") -> str:
        """Look up a provision by its exact section number.

        Args:
            section: Section number alone, e.g. "180", "1043A", "601FC". Do not
                include the word "section" or a subsection in brackets.
            act: Optional Act name to narrow the search, e.g. "corporations",
                "privacy", "criminal code".
        """
        hits = find_section(corpus, section, act=act or None)
        # Log every call so the run can be reviewed afterwards
        calls.append({"tool": "find_section", "args": {"section": section, "act": act},
                      "hits": len(hits)})
        # Tell the agent what to try next rather than return nothing
        if not hits:
            return (f"No section {section} found{' in ' + act if act else ''}. "
                    f"Check the number, or try search_text.")
        # Save the full text as evidence, and show the agent only an excerpt of up to three matches
        out = []
        for h in hits[:3]:
            keep(h["text"], f"{h['act']}.txt#s{h['section']}")
            out.append(f"[{h['title']} s{h['section']} — {h['heading']}] "
                       f"(line {h['start_line']})\n{h['text'][:AGENT_EXCERPT_CHARS]}...")
        return "\n\n".join(out)

    @tool(name="search_text")
    def search_text_tool(query: str, act: str = "") -> str:
        """Search for a literal string, with the following lines included.

        Use for penalties, maximum terms, amounts, defined terms and
        cross-references — anything tabular or structural that meaning-based
        search cannot reach.

        Args:
            query: Exact text to find, e.g. "Subsection 1043A(1)" to locate that
                row in a Schedule penalty table.
            act: Optional Act name to narrow the search.
        """
        hits = search_text(corpus, query, act=act or None, max_hits=AGENT_KEYWORD_HITS)
        calls.append({"tool": "search_text", "args": {"query": query, "act": act},
                      "hits": len(hits), "of": hits[0]["of"] if hits else 0})
        if not hits:
            # Hint at the most common cause of no match: an abbreviation the Act spells out in full
            return (f"No literal match for {query!r}. The corpus uses the Act's own "
                    f"wording — spell abbreviations out in full (e.g. 'Australian "
                    f"Privacy Principle 11' rather than 'APP 11'), or try "
                    f"find_section if you know the number.")
        out = []
        for h in hits:
            keep(h["text"], f"{h['act']}.txt#L{h['start_line']}")
            out.append(f"[{h['title']} line {h['start_line']} "
                       f"(hit {h['hit']} of {h['of']})]\n{h['text'][:AGENT_EXCERPT_CHARS]}")
        return "\n\n".join(out)

    @tool
    def semantic_search(query: str) -> str:
        """Find provisions by meaning rather than exact wording.

        Args:
            query: A description of the legal concept, in plain words.
        """
        try:
            # Fetch RETRIEVE_WIDE candidates from the knowledge base by vector (meaning-based) search
            raw = rt.retrieve(
                knowledgeBaseId=kb_id,
                retrievalQuery={"text": query},
                retrievalConfiguration={
                    "vectorSearchConfiguration": {"numberOfResults": RETRIEVE_WIDE}},
            ).get("retrievalResults", [])
        except Exception as e:
            # If the knowledge base call fails, steer the agent to the other tools
            calls.append({"tool": "semantic_search", "args": {"query": query},
                          "error": type(e).__name__})
            return "Semantic search is unavailable; use find_section or search_text."
        if not raw:
            calls.append({"tool": "semantic_search", "args": {"query": query}, "hits": 0})
            return "No matching provisions found."
        # Rerank the candidates and keep the best AGENT_SEARCH_TOP_K
        kept, moves = rerank_candidates(rt, query, raw, AGENT_SEARCH_TOP_K)
        calls.append({"tool": "semantic_search", "args": {"query": query},
                      "hits": len(kept),
                      "best_relevance": moves[0]["relevance"] if moves else None})
        out = []
        for m, c in zip(moves, kept):
            uri = c.get("location", {}).get("s3Location", {}).get("uri", "?")
            keep(c["content"]["text"], uri)
            # Collapse whitespace so the excerpt is compact
            body = " ".join(c["content"]["text"].split())
            # Label each passage with its Act's filename and relevance score
            out.append(f"[{uri.rsplit('/', 1)[-1].replace('.txt', '')}, "
                       f"relevance {m['relevance']:.3f}]\n{body[:AGENT_EXCERPT_CHARS]}...")
        return "\n\n".join(out)

    @tool(name="read_around")
    def read_around_tool(act: str, line: int) -> str:
        """Read the lines surrounding a position, to follow a reference.

        Args:
            act: Act name, as reported by another tool.
            line: Line number reported by find_section or search_text.
        """
        got = read_around(corpus, act, line)
        calls.append({"tool": "read_around", "args": {"act": act, "line": line},
                      "hits": 1 if got else 0})
        if not got:
            return f"Could not read around line {line} of {act!r}."
        keep(got["text"], f"{got['act']}.txt#L{got['from_line']}")
        # Show twice the usual excerpt length, since following a reference needs more context
        return (f"[{got['title']} lines {got['from_line']}-{got['to_line']}]\n"
                f"{got['text'][:AGENT_EXCERPT_CHARS * 2]}")

    # semantic_search needs the knowledge base
    tools = [find_section_tool, search_text_tool, read_around_tool]
    if kb_id:
        tools.insert(2, semantic_search)
    return tools


# Build the tools once with throwaway stores, just to show what the agent sees
_probe = build_toolset(KNOWLEDGE_BASE_ID, REGION, CORPUS or {}, {}, [])
print(f"✓ {len(_probe)} tools defined. This is what the agent sees:\n")
if not KNOWLEDGE_BASE_ID:
    print("  (semantic_search omitted — no knowledge base. Three modalities, and")
    print("   the one Module 5 already has is the missing one.)\n")
for _t in _probe:
    # Each tool's spec holds its name, arguments and description, taken from the function and its docstring
    _spec = _t.tool_spec
    _args = ", ".join(_spec["inputSchema"]["json"]["properties"])
    print(f"  {_spec['name']}({_args})")
    print(f"      {_spec['description'].splitlines()[0]}")

## Step 7: Inspect One Agent Investigation

`run_agent` runs the agent on one question and collects the evidence it finds. The
same function is used for scoring in Step 9.

In [ ]:
# Run the agent on one question and collect the evidence it finds
DEMO_QUESTION = ("What is the maximum penalty for insider trading under "
                 "section 1043A of the Corporations Act 2001 (Cth)?")

# The agent (Nova Pro) only searches. The passages it collects are later given
# to the model under test, which writes the answer
async def run_agent(question, kb_id, region, corpus):
    """Run one investigation. Returns (spans_for_prompt, record)."""
    # Each tool writes the passages it finds into evidence and logs itself in calls
    evidence, calls = {}, []
    prompt = AGENT_SYSTEM_PROMPT
    # Without a knowledge base, drop the semantic_search line from the instructions
    if not kb_id:
        prompt = "\n".join(l for l in prompt.split("\n") if not l.startswith("- semantic_search("))
    # temperature=0 keeps the agent's choices as repeatable as possible,
    # and callback_handler=None stops it streaming its output into the notebook
    agent = Agent(model=BedrockModel(region_name=region, model_id=ORCHESTRATOR_MODEL,
                                     temperature=0.0, max_tokens=1024),
                  tools=build_toolset(kb_id, region, corpus, evidence, calls),
                  system_prompt=prompt, callback_handler=None)

    record = {"question": question[:160], "status": "ok"}
    # The turn limit stops the agent searching forever. An error is recorded
    # rather than stopping the whole evaluation
    try:
        result = await agent.invoke_async(question + "\n\nFind the provisions needed to answer this.",
                                          limits=Limits(turns=AGENT_MAX_TURNS))
        record["brief"] = str(result).strip()[:300]
        record["orchestrator_tokens"] = dict(result.metrics.accumulated_usage or {})
    except Exception as e:
        record["status"] = f"error: {type(e).__name__}"
    # Flag runs where the agent replied without searching at all
    if not calls and record["status"] == "ok":
        record["status"] = "no_search"

    # Keep evidence in the order found, up to the budget
    spans, used = [], 0
    for text, span in evidence.items():
        if used + len(text) <= AGENT_CONTEXT_CHARS:
            spans.append(span)
            used += len(text)

    # Summarise which tools were used and how much evidence made it into the prompt
    tools_used = sorted({c["tool"] for c in calls})
    record.update({"calls": calls, "tool_calls": len(calls), "tools_used": tools_used,
                   "modalities": len(tools_used), "spans_in_prompt": len(spans), "context_chars": used})
    return spans, record

print("✅ Ready to proceed!")

**Run the agent on one question**, the s1043A penalty question.

> Things to look at in the output:
>
> 1. Which tools the agent called, and in what order.
> 2. Whether `Does the answering prompt contain the penalty?` is `True`. If it is,
>    the agent found the penalty text.
> 3. How many kinds of search it used. RAG always uses one.
>
> The agent can take a different path each run, even at temperature 0.

In [ ]:
# Run the agent on the demo question and print what it did
# asyncio.run runs the async function to completion (nest_asyncio, set up in Settings, allows this in Jupyter)
spans, rec = asyncio.run(run_agent(DEMO_QUESTION, KNOWLEDGE_BASE_ID, REGION, CORPUS))

print(f"Question: {DEMO_QUESTION}\n\nWhat the agent did:")
# List each tool call with its arguments (empty ones left out) and how many results it returned
for i, c in enumerate(rec["calls"], 1):
    args = ", ".join(f"{k}={v!r}" for k, v in c["args"].items() if v != "")
    print(f"  {i}. {c['tool']}({args}) → {c.get('error') or str(c.get('hits', 0)) + ' hit(s)'}")
print(f"\nKinds of search used: {rec['modalities']} ({', '.join(rec['tools_used'])})")
print(f"Evidence collected: {rec['spans_in_prompt']} passages, {rec['context_chars']:,} characters")
# Check whether the answering prompt would include the 15-year penalty for insider trading
print(f"Does the evidence contain the penalty? {'15 years imprisonment' in build_grounded_prompt(spans, '')}")

## Step 8: Define the Agent Solver

The cell below wraps the agent as an Inspect solver, so it can be scored the same
way as the other configurations. It only defines the solver.

In [ ]:
# Wrap the agent as an Inspect solver
from inspect_ai.solver import solver, Generate, TaskState


# A solver is the Inspect AI step that produces each answer. This one runs the agent first,
# then asks the model under test to answer using the evidence the agent found
def make_agentic_solver(kb_id, region, corpus, agent_log):
    """Agent investigates with four modalities; MODEL_UNDER_TEST writes the answer."""

    @solver
    def agentic_solver():
        async def solve(state: TaskState, generate: Generate) -> TaskState:
            # Run the agent on the question and keep its record for the report later
            question = state.user_prompt.text
            spans, record = await run_agent(question, kb_id, region, corpus)
            agent_log.append(record)
            # Swap the question for the grounded prompt. If the agent found nothing, the model gets the bare question
            if spans:
                state.user_prompt.text = build_grounded_prompt(spans, question)
            # generate() sends the prompt to the model under test
            return await generate(state)
        return solve

    return agentic_solver()


print("✓ Solver defined. Same dataset, rubric and judge as Modules 2 and 5;")
print("  the only thing that changes is how the evidence was found.")

## Step 9: Score It

The cell below scores the agent on the 50 questions, using the same rubric and judge.

> This takes about 6–12 minutes to run. Once it finishes, read the next cell before
> looking at the score.

In [ ]:
# Score the agent on the 50 questions
# agentic_log collects one record per question from the agent
agentic_log = []
ANSWERER = MODEL_UNDER_TEST
AGENTIC_CELL = "agentic"

print(f"Scoring {AGENTIC_CELL} on {NUM_EVAL_RECORDS} questions (answerer: {ANSWERER})...")
# Score all 50 questions and save the agent's settings with the result
agentic_score, agentic_per_item = score_cell(
    AGENTIC_CELL, make_agentic_solver(KNOWLEDGE_BASE_ID, REGION, CORPUS, agentic_log), model=ANSWERER,
    config={"framework": "strands", "orchestrator_model": ORCHESTRATOR_MODEL, "answerer": ANSWERER,
            "semantic_search_available": bool(KNOWLEDGE_BASE_ID), "max_turns": AGENT_MAX_TURNS,
            "context_chars": AGENT_CONTEXT_CHARS})

**Inspect what the agent did.** Read this before the score. The cell below
summarises the agent's searches across the 50 questions.

**The key line is "Questions using a NON-semantic tool".** If it's 0, the agent
only ever searched by meaning, which is the same as RAG with reranking at extra
cost. It also reports questions where the agent didn't search at all, and how many
tokens the agent used.

In [ ]:
# Report what the agent did across the 50 questions
from collections import Counter
n = len(agentic_log)
# Questions where the agent used at least one tool other than semantic_search
non_semantic = sum(bool(set(e["tools_used"]) - {"semantic_search"}) for e in agentic_log)
# Add up the agent's token usage across all questions
tokens = Counter()
for e in agentic_log:
    tokens.update(e.get("orchestrator_tokens") or {})

print(f"Tool calls:                          {dict(Counter(c['tool'] for e in agentic_log for c in e['calls']))}")
print(f"Questions using >1 kind of search:   {sum(e['modalities'] > 1 for e in agentic_log)}/{n}")
print(f"Questions using a NON-semantic tool: {non_semantic}/{n}")
print(f"Status:                              {dict(Counter(e['status'] for e in agentic_log))}")
print(f"Agent tokens:                        {tokens['inputTokens']:,} in / {tokens['outputTokens']:,} out")
if non_semantic == 0:
    print("\n⚠️  The agent only ever searched by meaning, so this is Module 5 with extra cost.")

## Step 10: Compare Against Every Other Configuration

The cell below combines every full 50-question result in S3 into one table. A `—`
means not measured yet.

> Compare the agentic search score with each earlier configuration.

In [ ]:
# Print every configuration's score side by side
# Each entry is (saved configuration name, label to print)
cell_order = [("none", "No retrieval"), ("rag", "+ RAG (top 10)"),
              ("rag-rerank", "+ RAG + reranking"), ("agentic", "+ Agentic search")]
latest = print_comparison(cell_order)

## Step 11: What Did Agentic Search Change?

The cell below compares agentic search with RAG with reranking, question by question: how many it fixed, how many it broke, and
whether that split could be chance.

> Look at the fixed and broken counts and whether `p < 0.05`. At 50 questions a
> small gain can be noise, so the questions that changed matter more than the total.

In [ ]:
# Compare agentic search with RAG and reranking, question by question
# The newest saved runs of RAG with reranking (Module 5) and of agentic search
base_rec = latest.get(("rag-rerank", MODEL_LABEL))
var_rec = latest.get((AGENTIC_CELL, MODEL_LABEL))

if base_rec and var_rec:
    # Count questions the agent fixed or broke compared with reranking
    r = paired_compare(base_rec["per_item"], var_rec["per_item"])
    print(f"RAG + reranking: {base_rec['score']:.3f}    {AGENTIC_CELL}: {var_rec['score']:.3f}\n")
    print(f"  Fixed  {len(r['fixed']):>3}  (reranking wrong, agent right)")
    print(f"  Broke  {len(r['broken']):>3}  (reranking right, agent wrong)")
    print(f"  Same   {len(r['both_right']) + len(r['both_wrong']):>3}")
    # p below 0.05 means a gap this size would rarely happen by chance
    print(f"\n  McNemar p = {r['p_value']:.3g}: "
          f"{'unlikely to be chance' if r['p_value'] < 0.05 else 'could be chance at 50 questions'}")

    # Score by category, before and after
    var_by_id = {str(p["id"]): p["score"] for p in var_rec["per_item"]}
    by_cat = {}
    # For each category: number of questions, correct with reranking, correct with the agent
    for p in base_rec["per_item"]:
        c = by_cat.setdefault(p["category"], [0, 0, 0])
        c[0] += 1; c[1] += p["score"]; c[2] += var_by_id.get(str(p["id"]), 0)
    print("\n  By category (RAG + reranking → agentic search):")
    for cat, (n_cat, b, v) in sorted(by_cat.items()):
        print(f"    {cat:<24} {b / n_cat:.2f} → {v / n_cat:.2f}")
else:
    print("Needs full-size results for both RAG with reranking (Module 5) and agentic search (Step 9).")

---

## Success Criteria

Before moving on, verify that you have:

- [ ] Loaded 12 Acts from S3 with 0 non-breaking spaces remaining (Step 3)
- [ ] Seen `search_text` find the `Subsection 1043A(1)` penalty row (Step 5)
- [ ] Checked which kinds of search the agent used on the demo question (Step 7)
- [ ] Scored agentic search on all 50 questions (Step 9)
- [ ] Checked how many questions used a non-semantic tool (Step 9)
- [ ] Read the fixed/broken split and its p-value (Step 11)

---

## Key Takeaways

- **RAG is one kind of search.** Searching by meaning can't look up a section number,
  match an exact phrase, or follow a cross-reference.
- **Reranking improves the order, but can't find what search missed.** A bare
  penalty-table row has nothing for meaning-based search to match.
- **An agent helps when choosing the search is the hard part.**
- **A tool is a function plus a decorator.** Strands builds the interface from the
  type hints and docstring.
- **Keep the answering model fixed**, or you're testing a different model, not a
  different search.
- **Invisible characters can break search.** Non-breaking spaces would have hidden
  exactly the rows the exact-phrase search is for.

---

## Next Steps

- **Add a tool**, such as one that looks up defined terms. Each is one function.